# Microbatch Repartitioning 

Notebook to run operations to support ticket: https://github.com/cal-itp/data-infra/issues/4865. Specifically, operations related to copying an existing (non-partitioned, non-microbatch) table and making a new partitioned version in the original location to transition a model to microbatch "in-place".

To run this, you need to auth: `gcloud auth application-default login --login-config=../../../iac/login.json` and select the Python kernel associated with this Poetry env. 

In [ ]:
# enter the model to be re-partitioned 
# format: project.schema.table
MODEL_TO_PROCESS = 'cal-itp-data-infra-staging.laurie_mart_gtfs.dim_stop_times'

# enter the partition info
PARTITION_COLUMN = '_feed_valid_from'

In [ ]:
split_model = MODEL_TO_PROCESS.split('.')
project = split_model[0]
schema = split_model[1]
table_name = split_model[2]

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project)

In [ ]:
# first, copy the current state of the model 

orig_table_copy_name = project + '.' + schema + '.' + table_name + '_orig'

job = client.copy_table(MODEL_TO_PROCESS, orig_table_copy_name)
job.result()

# check that row counts are equal 
orig_row_count = client.query(f"select count(*) from {MODEL_TO_PROCESS}").result()
copy_row_count = client.query(f"select count(*) from {orig_table_copy_name}").result()

assert next(orig_row_count) == next(copy_row_count), "copy and original should have same row count"

print(f"Copied {MODEL_TO_PROCESS} into {orig_table_copy_name}")

In [ ]:
# now, take everything from the copy and partition it

partitioned_table = project + '.' + schema + '.' + table_name + '_partitioned'

job_config = bigquery.QueryJobConfig()
job_config.destination = partitioned_table
job_config.time_partitioning = bigquery.table.TimePartitioning(type_ = 'DAY', field = PARTITION_COLUMN)

query = f"select * from {orig_table_copy_name}"

client.query_and_wait(query, job_config=job_config)

# check that row counts are equal 
orig_row_count = client.query(f"select count(*) from {MODEL_TO_PROCESS}").result()
partitioned_row_count = client.query(f"select count(*) from {partitioned_table}").result()

assert next(orig_row_count) == next(partitioned_row_count), "copy and partitioned version should have same row count"

print(f"Copied data into partitioned version model {partitioned_table}")


In [ ]:
# now, delete the original 
client.delete_table(MODEL_TO_PROCESS)

print(f"Deleted {MODEL_TO_PROCESS}")

In [ ]:
# now copy the partitioned version into the original location

job = client.copy_table(partitioned_table, MODEL_TO_PROCESS)
job.result()

# check that row counts are equal 
partitioned_copied_row_count = client.query(f"select count(*) from {MODEL_TO_PROCESS}").result()
copy_row_count = client.query(f"select count(*) from {orig_table_copy_name}").result()

assert next(partitioned_copied_row_count) == next(copy_row_count), "partitioned version and copy of original should have same row count"

print(f"Copied {partitioned_table} into {MODEL_TO_PROCESS}")

In [ ]:
# now, delete the partitioned copy 
client.delete_table(partitioned_table)

print(f"Deleted {partitioned_table}")